# 24. Retrieval with a Feedback Loop

Learn from user ratings: boost chunks that helped on similar past questions, demote those that did not.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

A static retriever never learns. Users constantly give signals - thumbs up/down, clicks, corrections. A **feedback loop** stores those signals and lets them nudge future rankings: chunks that helped on similar past questions get a boost, chunks that misled get demoted.

## How it works

1. Log (query, chunk, rating) after each answer
2. Persist feedback (here a JSON file, in production a database)
3. At query time compute similarity between the new query and past queries
4. Add `boost x rating x similarity` to each chunk's score
5. Rank by the adjusted score

```text
 user rates answer --> feedback store (query, chunk, +1/-1)
 new query --> base score + sum( rating x sim(new query, past query) ) --> re-ranked results
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 10:55:12.736000 8222 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x74910f0000>
    label = <none> 
    device = <AGXG13GDevice: 0x7485fa0000>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x7490fe8000>
        label = <none> 
        device = <AGXG13GDevice: 0x7485fa0000>
            name = Apple M1 
    retainedReferences = 1


19 chunks indexed


In [2]:
FEEDBACK_FILE = DATA / "feedback.json"
feedback = json.loads(FEEDBACK_FILE.read_text()) if FEEDBACK_FILE.exists() else []

def record_feedback(query, chunk_idx, rating):
    """rating: +1 helpful, -1 not helpful"""
    feedback.append({"query": query, "chunk": chunk_idx, "rating": rating})
    FEEDBACK_FILE.write_text(json.dumps(feedback, indent=2))

At query time, each past rating is weighted by how similar its question is to the new one.

In [3]:
import numpy as np

def search_with_feedback(query, k=3, boost=0.15):
    scores = store.vecs @ embed(query)[0]
    if feedback:
        past = embed([f["query"] for f in feedback]) @ embed(query)[0]
        for f, sim in zip(feedback, past):
            if sim > 0.5:
                scores[f["chunk"]] += boost * f["rating"] * sim
    return [{"idx": int(i), "text": chunks[i], "score": float(scores[i]), "meta": {}} for i in np.argsort(-scores)[:k]]

query = "How do I measure the quality of retrieval?"
before = search_with_feedback(query)
show(before, width=90)

0.591  user or group identifiers. Running models locally with a tool such as Ollama keeps sensiti...
0.560  delay.  ## Security and Privacy  Retrieved text is untrusted input. A malicious document c...
0.560  ten vague, short or badly phrased, so the query is rewritten before retrieval. Query rewri...


Simulate a user marking the top hit as unhelpful and the third as helpful, then ask a *similar* question.

In [4]:
record_feedback(query, before[0]["idx"], -1)
record_feedback(query, before[2]["idx"], +1)
print("--- after feedback, similar query")
show(search_with_feedback("How can retrieval quality be evaluated?"), width=90)

--- after feedback, similar query


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x74910f0380>
    label = <none> 
    device = <AGXG13GDevice: 0x7485fa0000>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x7490fe8000>
        label = <none> 
        device = <AGXG13GDevice: 0x7485fa0000>
            name = Apple M1 
    retainedReferences = 1


0.596  user or group identifiers. Running models locally with a tool such as Ollama keeps sensiti...
0.564  delay.  ## Security and Privacy  Retrieved text is untrusted input. A malicious document c...
0.564  is usually measured with cosine similarity, which compares direction rather than length. I...


In production, store feedback in a database, decay old ratings and use the signal to fine-tune the retriever.

## Rank movement table
Show exactly how the ratings moved each chunk.

In [5]:
import pandas as pd
q2 = "How can retrieval quality be evaluated?"
base = {h["idx"]: r for r, h in enumerate(store.search(q2, 6), 1)}
adj = {h["idx"]: r for r, h in enumerate(search_with_feedback(q2, k=6), 1)}
pd.DataFrame([{"chunk": i, "rank before": base.get(i), "rank after": adj.get(i), "text": chunks[i][:60]} for i in set(base) | set(adj)]).sort_values("rank after")

,chunk,rank before,rank after,text
1,8,5.0,1.0,"ten vague, short or badly phrased, so the quer..."
5,17,2.0,2.0,delay.\n\n## Security and Privacy\n\nRetrieved...
0,2,3.0,3.0,"is usually measured with cosine similarity, wh..."
2,11,4.0,4.0,"them, because the model can ignore the context..."
4,13,6.0,5.0,uch as what the main themes of a corpus are. P...
3,12,NaN,6.0,nowledge graph stores facts as triples of subj...
6,18,1.0,NaN,user or group identifiers. Running models loca...


## Feedback only affects *similar* queries
An unrelated query should be unaffected.

In [6]:
unrelated = "How do I secure my RAG system against prompt injection?"
print("same ranking with and without feedback:",
      [h["idx"] for h in store.search(unrelated, 3)] == [h["idx"] for h in search_with_feedback(unrelated, 3)])

same ranking with and without feedback: False


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x74910f0380>
    label = <none> 
    device = <AGXG13GDevice: 0x7485fa0000>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x7490fe8000>
        label = <none> 
        device = <AGXG13GDevice: 0x7485fa0000>
            name = Apple M1 
    retainedReferences = 1


## Guard against feedback abuse
One user spamming ratings should not dominate. Cap the total boost per chunk.

In [7]:
import numpy as np
def search_capped(query, k=3, boost=0.15, cap=0.3):
    scores = store.vecs @ embed(query)[0]
    delta = np.zeros_like(scores)
    if feedback:
        sims = embed([f["query"] for f in feedback]) @ embed(query)[0]
        for f, s in zip(feedback, sims):
            if s > 0.5: delta[f["chunk"]] += boost * f["rating"] * s
    scores = scores + np.clip(delta, -cap, cap)
    return [{"idx": int(i), "text": chunks[i], "score": float(scores[i]), "meta": {}} for i in np.argsort(-scores)[:k]]

show(search_capped(q2), width=80)

0.713  ten vague, short or badly phrased, so the query is rewritten before retrieval. Q...
0.564  delay.  ## Security and Privacy  Retrieved text is untrusted input. A malicious ...
0.564  is usually measured with cosine similarity, which compares direction rather than...


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Learns from real usage | Needs enough traffic to be useful |
| Cheap: arithmetic on scores | Vulnerable to biased or malicious ratings |
| Easy to explain and audit | Popular chunks can crowd out better ones (feedback loops reinforce themselves) |

## When to use it

Products with real users and a thumbs up/down UI, ideally combined with periodic offline retraining of the retriever.

## Experiments to try

- Decay old feedback by age.
- Use clicks as implicit positive feedback.
- Train a small re-ranker from the logged pairs.

## Key takeaways

- Log everything users tell you.
- Weight feedback by query similarity.
- Cap and decay to stay robust.

---
**Next:** [25. Adaptive Retrieval](../25_adaptive_retrieval/adaptive_retrieval.ipynb)